# Strategic GNE selection — original bilateral P2P game

**Change `DATE` in the first code cell and Run All.** The notebook finds your EXISTING simulation tree before writing, performs/reloads the audited 24-hour / 1-hour reduction, **audits player costs at the start**, and searches for fair GNEs using the original bilateral constraints. No ParametricDAQP critical-region enumeration is needed; the separate dynect library remains the quadratic-game adapter.

We keep original `distributed.json`, `centralized.json`, `no_trade.json` and `fairness.json` untouched. New results are saved under `structures/<structure>/results/<DATE>/fair_gne/1h/`, where `<structure>` comes from `config.py` (e.g. `ieee33_4_cyclic`), next to the existing simulations. If the existing tree cannot be located unambiguously, the notebook stops rather than making a parallel one.

The 1-hour time aggregation is an approximation. Local duplicate cleanup, certified feeder screening and optional local-LP pruning are audited separately. Exact GNE claims apply only to the **saved reduced game**, with regularization set to zero by default.

## 0. Choose the date, find the *existing* simulation tree and test saving

In [ ]:
from pathlib import Path
import sys, json, time, hashlib
import numpy as np
import pandas as pd
from IPython.display import display

HERE=Path.cwd().resolve()
ROOT=next((p for p in (HERE,*HERE.parents) if (p/'scenarios.py').is_file()),None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook inside the ORIGINAL P2P repository (must find scenarios.py).')
DYNECT_REPO=Path.home()/'Documents/Github/dynect_py_p2p'
for folder in (ROOT, HERE, DYNECT_REPO):
    if folder.is_dir() and str(folder) not in sys.path:
        sys.path.insert(0,str(folder))

# Require this module to come from the current notebook's directory.
import importlib.util
module_spec=importlib.util.find_spec('strategic_GNE_selection')
expected_module=(HERE/'strategic_GNE_selection.py').resolve()
if not expected_module.is_file():
    raise FileNotFoundError(f'Strategic suite missing next to notebook: {expected_module}')
if module_spec is None or Path(module_spec.origin).resolve()!=expected_module:
    raise ImportError(f'Wrong strategic suite imported; expected {expected_module}, '
                      f'found {None if module_spec is None else module_spec.origin}. '
                      'Restart kernel and run this notebook from its own folder.')
from strategic_GNE_selection import (
    StrategicGame, ensure_reduced_scenario, scenario_paths,
    find_simulation_root, test_results_write_access,
    reduction_counts, load_reduced_model, fairness_metrics,
    run_or_load_result, save_candidate, discover_prior_simulations,
    build_reduced_model, model_max_violation)
# The adapter lives next to this notebook in strategic_GNE_selection/p2p/.
import importlib.util
adapter_path=(HERE/'p2p'/'setup.py').resolve()
if not adapter_path.is_file():
    raise FileNotFoundError(
        f'P2P adapter missing: {adapter_path}\n'
        'Expected the p2p/ package inside strategic_GNE_selection/.')
adapter_spec=importlib.util.find_spec('p2p.setup')
if adapter_spec is None or Path(adapter_spec.origin).resolve()!=adapter_path:
    raise ImportError(
        'The active Python interpreter is importing a different p2p.setup. '
        f'Expected {adapter_path}; got {None if adapter_spec is None else adapter_spec.origin}. '
        'Restart your notebook kernel and remove old extra PYTHONPATH entries.')
from p2p.setup import build_game, load_grid, evaluated_costs

# ================ CHANGE THESE SETTINGS ======================
DATE='2016-06-15'               # Switch days here, then Restart/Run All
SIMULATION_ROOT=None            # Uses ROOT/structures/; override only for another real existing root
SOURCE_DT=0.5
TARGET_DT=1.0
SEED=42
RUN_LOCAL_LP_PRUNING=True
LP_REDUNDANCY_TOL=1e-9
CLEARING_EPSILON_KW=0.0        # exact bilateral reciprocity
INPUT_REGULARIZATION=0.0       # original model objective preferred
STATE_REGULARIZATION=0.0
FORCE_REBUILD=False             # re-run reductions if scenario CSV/data changed
FORCE_RECOMPUTE_RESULTS=False

RUN_MINIMAX=True
RUN_MAXMIN_SAVINGS=True
RUN_WEIGHTS=True
WEIGHT_SAMPLES=16
WEIGHT_LOG_STD=1.0             # larger => more aggressive reweighting
RUN_NASH=True
NASH_MAX_ITER=200
NASH_FW_TOL=1e-4
RUN_TRADE_SEARCH=True
TRADE_SAMPLES=10
TRADE_SIGMA_KW=0.15
RUN_PARETO_REFINEMENT=True
# =============================================================

import config
STRUCTURE=config.structure_name()   # follows the config.py switch
N_PROSUMERS=config.PROSUMER_SCENARIO
SIMULATION_ROOT=find_simulation_root(ROOT,structure_name=STRUCTURE,
                                     override=SIMULATION_ROOT,cwd=HERE)
paths=scenario_paths(SIMULATION_ROOT,DATE,target_dt=TARGET_DT,structure_name=STRUCTURE)
verified_fair_path=test_results_write_access(paths)
print('PROJECT ROOT:             ',ROOT)
print('MODULE ACTUALLY IMPORTED: ',__import__('strategic_GNE_selection').__file__)
print('EXISTING SIMULATION ROOT: ',SIMULATION_ROOT)
print('EXISTING RESULTS TREE:    ',paths['simulation_dir'].parent)
print('NEW FAIR RESULTS FOLDER:  ',verified_fair_path)
print('WRITE/READ/DELETE TEST:    PASSED')
print('Original simulation files found:')
prior=discover_prior_simulations(paths)
if prior: display(pd.DataFrame(prior)[['kind','path','json_valid']])
else: print('  No existing files on this particular day; existing results root verified.')

## 1. Reload or build the dated reduction; verify the files

In [ ]:
start=time.perf_counter()
case,audit,info=ensure_reduced_scenario(
    DATE,base_dir=SIMULATION_ROOT,structure_name=STRUCTURE,seed=SEED,
    source_dt=SOURCE_DT,target_dt=TARGET_DT,
    lp_prune=RUN_LOCAL_LP_PRUNING,lp_tol=LP_REDUNDANCY_TOL,
    clearing_epsilon_kw=CLEARING_EPSILON_KW,
    input_regularization=INPUT_REGULARIZATION,
    state_regularization=STATE_REGULARIZATION,
    force_rebuild=FORCE_REBUILD)
MODEL_PATH=Path(info['model_path'])
CACHE_DIR=Path(info['cache_dir'])
OUT_DIR=Path(info['results_dir'])
for folder in (CACHE_DIR,OUT_DIR,OUT_DIR/'candidates',OUT_DIR/'diagnostics'):
    folder.mkdir(parents=True,exist_ok=True)
assert MODEL_PATH.is_file() and Path(info['audit_path']).is_file()
loaded_case, loaded_audit=load_reduced_model(MODEL_PATH,check_matrices=True)
assert loaded_audit['date']==DATE and loaded_case.N==N_PROSUMERS and loaded_case.T==24
assert MODEL_PATH.resolve().is_relative_to(verified_fair_path)
print('Reduction:', 'new build' if info['rebuilt'] else 'cached',
      'time [s]:',round(time.perf_counter()-start,2))
print('Saved model [absolute]:',MODEL_PATH.resolve(),
      f'({MODEL_PATH.stat().st_size/1e6:.2f} MB)')
print('Saved audit [absolute]:',Path(info['audit_path']).resolve())
print('Reloaded matrices + metadata: OK')
display(pd.DataFrame(audit['stages']))
print('Mixed EV-availability slots:',audit['mixed_ev_availability_slots'])
print('Feeder rows retained:',len(audit['keep_shared_labels']))

## 2. Audit separability and objective fidelity **before using fairness optimization**

The condensed pseudo-gradient must have zero off-player blocks; independent physical-cost evaluations must not change when only another player moves. We also test whether changes in the model quadratic cost match changes in reported EUR cost. The empirical perturbations can be infeasible and are **not** GNE deviations. The model has no regularization by default. If this fails, stop: positive-weight QPs and Pareto-to-GNE claims need revision.

In [ ]:
sg=StrategicGame(case)
print('Dimensions: decisions=',sg.n,' rows=',len(sg.b),
      ' players=',sg.N,'dt=',case.data['dt'])
report=sg.audit_costs(trials=4,amplitude=0.03)
print({k:v for k,v in report.items() if k!='rows'})
display(pd.DataFrame(report['rows']))
COST_AUDIT_PATH=OUT_DIR/'diagnostics'/'cost_audit.json'
COST_AUDIT_PATH.write_text(json.dumps(report,indent=2,allow_nan=False),encoding='utf8')
assert json.loads(COST_AUDIT_PATH.read_text())['date']==report['date']
if not report['passed']:
    raise RuntimeError('Cost separability / objective calibration / regularization audit FAILED. '
                       'Inspect cost_audit.json before claiming fair GNEs.')
print('COST AUDIT PASSED. Saved:',COST_AUDIT_PATH.resolve())

## 3. Baseline vGNE and independent best-response verification

In [ ]:
baseline,hit=run_or_load_result(
    sg,CACHE_DIR/'vGNE.npz',MODEL_PATH,
    method='baseline_equal_weight',options={'weights':[1.0]*case.N},
    solve=lambda:sg.solve_weighted(np.ones(case.N)),
    verify=True,force=FORCE_RECOMPUTE_RESULTS)
u_vgne=np.asarray(baseline['u'],float)
base_ver=sg.verify(u_vgne)  # always recompute BRs, even for cached baseline
print('Baseline source:', 'exact-model cache' if hit else 'new optimization')
print('GNE:',base_ver['is_numerical_GNE'], 'Feasibility:',sg.feasibility(u_vgne))
print('Original physical costs [EUR]:',np.round(sg.physical_costs(u_vgne),6))
print('Model/physical discrepancy [EUR]:',np.round(sg.model_cost_discrepancy(u_vgne),8))
display(pd.DataFrame(base_ver['players']))
assert base_ver['is_numerical_GNE'], 'Baseline failed unilateral verification'
assert np.max(np.abs(sg.model_cost_discrepancy(u_vgne)))<1e-4
candidates={
    'vGNE':dict(u=u_vgne,verification=base_ver,method='equal_weight_vGNE',meta={})
}

## 4. No-trade cost benchmark and actual trading volume

In [ ]:
nt_sol,hit=run_or_load_result(
    sg,CACHE_DIR/'no_trade.npz',MODEL_PATH,
    method='no_trade',options={'fixed_trades_kw':0.0},
    solve=lambda:sg.solve_no_trade(),force=FORCE_RECOMPUTE_RESULTS)
J_no_trade=np.asarray(nt_sol['physical_costs'],float)
J_vgne=sg.physical_costs(u_vgne)
print('No-trade source:', 'cache' if hit else 'solved')
print('Individual no-trade costs [EUR]:',np.round(J_no_trade,4))
print('Individual baseline gains [EUR]:',np.round(J_no_trade-J_vgne,4))
print('vGNE total gain [EUR]:',round((J_no_trade-J_vgne).sum(),5))
print('Trading volume abs [kWh]:',float(np.sum(np.abs(sg.extract_trades(u_vgne)))*TARGET_DT))
print('Metrics:')
display(pd.DataFrame([fairness_metrics(J_vgne,J_no_trade)]).drop(columns=['gains','proportional_gains']))

## 5. Direct fairness anchors: regularized minimax and max-min savings

These objectives remain convex because the individual quadratic costs are convex. A **positive sum-cost tie weight** makes the resulting optimization strictly increasing in individual costs: subject to a global optimum and separable costs, this produces a Pareto-efficient GNE. Unlike the old fragile two-stage lexicographic QCP, this uses a *single* optimization. It approximates rather than exactly solves the lexicographic minimax criterion.

In [ ]:
if RUN_MINIMAX:
    for kind in ('absolute_cost','absolute_loss','relative_loss'):
        print('\nMINIMAX',kind)
        opts={'no_trade_costs_eur':J_no_trade.tolist(),'kind':kind,'tie_weight':1e-4}
        try:
            sol,hit=run_or_load_result(
                sg,CACHE_DIR/f'minimax_{kind}.npz',MODEL_PATH,
                method='monotone_minimax',options=opts,
                solve=lambda kind=kind:sg.solve_robust_minimax(
                    J_no_trade,kind=kind,tie_weight=1e-4),
                verify=True,force=FORCE_RECOMPUTE_RESULTS)
            ver=sg.verify(sol['u'])
            name=f'minimax_{kind}'
            candidates[name]=dict(u=sol['u'],verification=ver,method=name,
                                  meta={'worst_metric':sol['worst_metric'],
                                        'tie_weight':1e-4})
            print('Cached:',hit,'worst metric:',sol['worst_metric'],
                  'GNE:',ver['is_numerical_GNE'])
        except (RuntimeError,ValueError) as exc:
            print('Skipped, solver not certified:',exc)

In [ ]:
if RUN_MAXMIN_SAVINGS:
    for relative in (False,True):
        name='maxmin_relative_savings' if relative else 'maxmin_absolute_savings'
        print('\n',name)
        try:
            sol,hit=run_or_load_result(
                sg,CACHE_DIR/f'{name}.npz',MODEL_PATH,
                method=name,
                options={'nt':J_no_trade.tolist(),'relative':relative,'tie_weight':1e-4},
                solve=lambda relative=relative:sg.solve_robust_maxmin_savings(
                    J_no_trade,relative=relative,tie_weight=1e-4),
                verify=True,force=FORCE_RECOMPUTE_RESULTS)
            ver=sg.verify(sol['u'])
            candidates[name]=dict(u=sol['u'],verification=ver,method=name,
                                  meta={'minimum_savings':sol['minimum_savings']})
            print('Min savings metric:',sol['minimum_savings'],
                  'GNE:',ver['is_numerical_GNE'])
        except (RuntimeError,ValueError) as exc: print('Skipped:',exc)

## 6. Strategic positive weights

For the separable original bilateral game, a global solution of a positive weighted sum of player costs is a GNE. Sample more aggressively with `WEIGHT_LOG_STD`; also test the Nash-derived marginal weights below. Not all weighted outcomes satisfy positive individual savings.

In [ ]:
weight_results=[]
if RUN_WEIGHTS:
    rng=np.random.default_rng(SEED)
    weights=[np.ones(case.N)]
    for _ in range(WEIGHT_SAMPLES):
        lw=rng.normal(0,WEIGHT_LOG_STD,case.N)
        lw-=lw.mean()
        weights.append(np.exp(lw))
    for k,w in enumerate(weights):
        try:
            sol,hit=run_or_load_result(
                sg,CACHE_DIR/f'weighted_{k:03d}.npz',MODEL_PATH,
                method='positive_weighted_potential',options={'weights':w.tolist()},
                solve=lambda w=w:sg.solve_weighted(w),
                verify=True,force=FORCE_RECOMPUTE_RESULTS)
            ver=sg.verify(sol['u'])
            name=f'weighted_{k:03d}'
            candidates[name]=dict(u=sol['u'],verification=ver,
                                  method='positive_weighted_GNE',meta={'weights':w.tolist()})
            met=fairness_metrics(sol['physical_costs'],J_no_trade)
            weight_results.append(dict(trial=k,weights=np.round(w,3).tolist(),
                                       cached=hit,GNE=ver['is_numerical_GNE'],
                                       nbs_log=met['nbs_log'],total_gain=met['total_gain']))
        except (RuntimeError,ValueError) as exc:
            weight_results.append(dict(trial=k,error=str(exc)))
    display(pd.DataFrame(weight_results).sort_values('nbs_log',ascending=False))

## 7. Nash bargaining and a certified weighted-equilibrium refinement

If any individual saving is nonpositive, log Nash product is 1ot admissible (`-inf` in the metrics table); that does *not* make other fairness metrics invalid. Frank–Wolfe iterates are retained as **diagnostic candidates** even when unverified. A subsequent weighted-QP solve from its marginal weights can yield a verified GNE; it is not a proof of globally optimal Nash bargaining.

In [ ]:
nash_sol=None
if RUN_NASH:
    opts={'no_trade_eur':J_no_trade.tolist(),'max_iter':NASH_MAX_ITER,
          'fw_gap_tol':NASH_FW_TOL}
    try:
        nash_sol,hit=run_or_load_result(
            sg,CACHE_DIR/'nash_bargaining.npz',MODEL_PATH,
            method='nash_fw',options=opts,
            solve=lambda:sg.solve_nash_bargaining(J_no_trade,max_iter=NASH_MAX_ITER,
                                                   fw_gap_tol=NASH_FW_TOL,verbose=True),
            verify=True,force=FORCE_RECOMPUTE_RESULTS)
        ver=sg.verify(nash_sol['u'])
        print('Nash: cached',hit,'converged',nash_sol['converged'],
              'FW gap',nash_sol['fw_gap'],'GNE?',ver['is_numerical_GNE'])
        # Keep an unconverged iterate, visibly labeled and never called a certified GNE.
        candidates['nash_FW_iterate']=dict(u=nash_sol['u'],verification=ver,
             method='unconverged_nash_iterate' if not nash_sol['converged'] else 'nash_FW',
             meta={'FW_gap':nash_sol['fw_gap'],'converged':nash_sol['converged']})
        if nash_sol.get('trace'):
            tracefile=OUT_DIR/'diagnostics'/'nash_trace.csv'
            pd.DataFrame(nash_sol['trace']).to_csv(tracefile,index=False)
        w=sg.nash_derived_weights(nash_sol['u'],J_no_trade)
        refined,hit=run_or_load_result(
            sg,CACHE_DIR/'nash_derived_weighted.npz',MODEL_PATH,
            method='nash_derived_weighted',options={'weights':w.tolist()},
            solve=lambda:sg.solve_weighted(w),verify=True,force=FORCE_RECOMPUTE_RESULTS)
        ver=sg.verify(refined['u'])
        candidates['nash_derived_weighted']=dict(u=refined['u'],verification=ver,
                           method='nash_derived_weighted_GNE',meta={'weights':w.tolist()})
        print('Nash marginal weights:',np.round(w,4))
        print('Weighted-refined GNE:',ver['is_numerical_GNE'],
              'log Nash score:',fairness_metrics(refined['physical_costs'],J_no_trade)['nbs_log'])
    except (ValueError,RuntimeError) as exc:
        print('Nash not available/certified for this scenario:',exc)

## 8. Pareto refinement of a chosen fair dispatch

Take any **jointly feasible** fairness target (including an approximate Nash iterate), cap every player's cost at the target's cost plus a small slack, and minimize a *positive weighted sum* over the original constraints. A globally solved refinement is Pareto-efficient in the original feasible game and therefore a GNE when player costs are separable. It need not preserve a nonmonotone fairness score such as variance exactly.

In [ ]:
if RUN_PARETO_REFINEMENT:
    anchor='nash_FW_iterate' if 'nash_FW_iterate' in candidates else (
        'minimax_absolute_loss' if 'minimax_absolute_loss' in candidates else 'vGNE')
    ref=np.asarray(candidates[anchor]['u'],float)
    print('Pareto refinement anchor:',anchor)
    try:
        # Cache-key the actual anchor model costs, not only the anchor label.
        ref_j=sg.model_costs(ref).tolist()
        sol,hit=run_or_load_result(
            sg,CACHE_DIR/'pareto_refinement.npz',MODEL_PATH,
            method='pareto_refine_cost_caps',
            options={'anchor_model_costs':ref_j,'cost_slack_eur':1e-5,
                     'weights':[1.0]*case.N},
            solve=lambda:sg.solve_pareto_refinement(ref,cost_slack_eur=1e-5),
            verify=True,force=FORCE_RECOMPUTE_RESULTS)
        ver=sg.verify(sol['u'])
        candidates['pareto_refined']=dict(u=sol['u'],verification=ver,
                                         method='pareto_cost_cap_refinement',
                                         meta={'anchor':anchor,'cost_caps':sol['cost_caps']})
        print('Pareto-refined GNE:',ver['is_numerical_GNE'])
        print('Change from anchor in individual costs [EUR]:',
              np.round(sg.physical_costs(sol['u'])-sg.physical_costs(ref),6))
    except (ValueError,RuntimeError) as exc: print('Pareto refinement skipped:',exc)

## 9. Search reciprocal trading schedules

Both directed trades are fixed to the same opposite-valued edge schedule. The remaining device variables are optimized subject to the full original constraints. Alternative schedules can yield Pareto-inefficient yet numerically verified GNEs; infeasible schedules are skipped.

In [ ]:
trades=[]
if RUN_TRADE_SEARCH:
    rng=np.random.default_rng(SEED)
    q0=sg.extract_trades(u_vgne)
    for k in range(TRADE_SAMPLES+1):
        q=q0.copy() if k==0 else q0+rng.normal(0,TRADE_SIGMA_KW,q0.shape)
        try:
            sol,hit=run_or_load_result(
                sg,CACHE_DIR/f'trade_{k:03d}.npz',MODEL_PATH,
                method='fixed_bilateral_schedule',options={'trades_kw':q.tolist()},
                solve=lambda q=q:sg.solve_weighted(fixed_trades=q),
                verify=True,force=FORCE_RECOMPUTE_RESULTS)
            ver=sg.verify(sol['u'])
            name=f'trade_{k:03d}'
            candidates[name]=dict(u=sol['u'],verification=ver,
                                  method='bilateral_trade_search',meta={'trial':k})
            met=fairness_metrics(sol['physical_costs'],J_no_trade)
            trades.append(dict(trial=k,status='solved',cached=hit,
                               rms_kw=float(np.sqrt(np.mean((q-q0)**2))),
                               GNE=ver['is_numerical_GNE'],total_gain=met['total_gain']))
        except (RuntimeError,ValueError) as exc:
            trades.append(dict(trial=k,status=f'failed: {exc}'))
    display(pd.DataFrame(trades))

## 10. Save ALL candidates and comparison metrics, clearly labeling failed GNE checks

In [ ]:
rows=[]
for name,rec in candidates.items():
    u=np.asarray(rec['u'],float).ravel()
    J=sg.physical_costs(u)
    met=fairness_metrics(J,J_no_trade)
    verified=bool(rec['verification']['is_numerical_GNE'])
    rows.append(dict(name=name,verified_GNE=verified,method=rec['method'],
        **{k:v for k,v in met.items() if k not in ('gains','proportional_gains')}))
    output=OUT_DIR/'candidates'/f'{name}.npz'
    save_candidate(output,case,u,method=rec['method'],
                   verification=rec['verification'],extra=rec.get('meta'))
    assert output.exists()
    with np.load(output,allow_pickle=False) as reload:
        assert np.allclose(reload['u'],u)
comparison=pd.DataFrame(rows).sort_values(['verified_GNE','nbs_log'],ascending=[False,False])
display(comparison)
CSV=OUT_DIR/'comparison.csv'
comparison.to_csv(CSV,index=False)
assert pd.read_csv(CSV).shape[0]==len(candidates)
manifest=dict(date=DATE,structure=STRUCTURE,resolution_h=TARGET_DT,
    source_resolution_h=SOURCE_DT,source_repo=str(ROOT),
    simulation_root=str(SIMULATION_ROOT),
    model_path=str(MODEL_PATH.resolve()),
    model_sha256=hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest(),
    reduction_audit_path=str(Path(info['audit_path']).resolve()),
    cost_audit_path=str(COST_AUDIT_PATH.resolve()),
    out_dir=str(OUT_DIR.resolve()),n_candidates=len(candidates),
    n_verified_GNE=int(comparison['verified_GNE'].sum()),
    unregularized=report['zero_regularization'],
    settings={'RUN_MINIMAX':RUN_MINIMAX,'RUN_MAXMIN_SAVINGS':RUN_MAXMIN_SAVINGS,
        'RUN_WEIGHTS':RUN_WEIGHTS,'WEIGHT_LOG_STD':WEIGHT_LOG_STD,
        'RUN_NASH':RUN_NASH,'RUN_TRADE_SEARCH':RUN_TRADE_SEARCH,
        'TRADE_SIGMA_KW':TRADE_SIGMA_KW,'RUN_PARETO_REFINEMENT':RUN_PARETO_REFINEMENT})
MANIFEST=OUT_DIR/'experiment_manifest.json'
MANIFEST.write_text(json.dumps(manifest,indent=2,allow_nan=False),encoding='utf8')
assert json.loads(MANIFEST.read_text())['date']==DATE
print('\nSAVED AND RELOADED:')
for p in (MODEL_PATH,Path(info['audit_path']),COST_AUDIT_PATH,CSV,MANIFEST):
    print(' -',p.resolve(),'size:',p.stat().st_size,'bytes')
print('Candidate files:',len(list((OUT_DIR/'candidates').glob('*.npz'))))

## 11. Independently verify any candidate from another method

`sg.verify(u)` holds all other players' *complete* decisions fixed but allows the chosen player's **all controls** to vary, enforcing the full shared constraints. With exact reciprocity, bilateral trades may be effectively locked; this is a property of the game, not an artificial BR restriction.

In [ ]:
VERIFY_NAME='vGNE'  # or 'nash_derived_weighted', 'pareto_refined', 'trade_000', etc.
assert VERIFY_NAME in candidates
verify_again=sg.verify(candidates[VERIFY_NAME]['u'],gap_tol_eur=1e-5)
print(VERIFY_NAME,'Verified GNE:',verify_again['is_numerical_GNE'])
display(pd.DataFrame(verify_again['players']))

## 12. Optional 30-minute reconfirmation (separate game)

A 1-hour GNE is **not automatically** a GNE of the original 30-minute game. This test lifts selected bilateral powers by holding them constant for both half-hours, reoptimizes device schedules under the full original 30-minute active-only model, and then checks unilateral best responses. It makes no claim about the original AC/reactive formulation.

In [ ]:
RUN_30MIN_CHECK=False
SELECT_CANDIDATE='vGNE'
if RUN_30MIN_CHECK:
    from scenarios import make_scenario_horizon
    fine_data=make_scenario_horizon(DATE,seed=SEED,resolution_h=0.5,lookahead_h=0)
    fine_case=build_game(fine_data,grid=load_grid(),clearing_epsilon_kw=0.0,
                         input_regularization=0.0,state_regularization=0.0)
    fine_game=StrategicGame(fine_case)
    q05=np.repeat(sg.extract_trades(candidates[SELECT_CANDIDATE]['u']),2,axis=1)
    fine_sol=fine_game.solve_weighted(fixed_trades=q05)
    fine_check=fine_game.verify(fine_sol['u'])
    print('30-min game verified GNE:',fine_check['is_numerical_GNE'])
    print('30-min individual costs:',fine_game.physical_costs(fine_sol['u']))
    full_out=Path(info['fair_dir'])/'0p5h_check'
    full_out.mkdir(parents=True,exist_ok=True)
    save_candidate(full_out/f'{SELECT_CANDIDATE}_reoptimized.npz',fine_case,
                   fine_sol['u'],method='30min_trade_lifted_reoptimized',
                   verification=fine_check,extra={'from_candidate':SELECT_CANDIDATE})

## Research interpretation

A feasible, globally solved Pareto-efficient allocation of this **separable-cost** game is a GNE. Other GNEs can be Pareto-inefficient because exact reciprocity constrains unilateral changes. The cost audit empirically checks separability and model/physical objective calibration; the formal claim still relies on the actual code's objective definition. All candidates are numerically checked via unilateral best responses. The original physical model (including reactive power/AC constraints) and finer-grid game require independent verification before claiming fidelity beyond the saved 24-hour/1-hour bilateral game.